In [ ]:
# Cell 1 - GPU/environment check
import platform
import subprocess
import sys

import torch

nvidia_smi = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
print(nvidia_smi.stdout or nvidia_smi.stderr)
print("Python:", sys.version.replace("\n", " "))
print("PyTorch:", torch.__version__)
print("torch.cuda.is_available():", torch.cuda.is_available())

if nvidia_smi.returncode != 0 or not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA is unavailable. In Colab select Runtime -> Change runtime type -> "
        "T4 GPU (or another GPU runtime), reconnect, and rerun this cell."
    )

gpu_properties = torch.cuda.get_device_properties(0)
print("GPU model:", torch.cuda.get_device_name(0))
print("Total VRAM (GiB):", round(gpu_properties.total_memory / 1024**3, 2))
print("PyTorch CUDA runtime:", torch.version.cuda)


In [ ]:
# Cell 2 - Project upload/extraction
import os
import shutil
import zipfile
from pathlib import Path

from google.colab import files

EXPECTED_ARCHIVE = "rocmpilot-research-evaluation-upgrade.zip"
print(f"Upload {EXPECTED_ARCHIVE} in the dialog.")
uploaded = files.upload()
if EXPECTED_ARCHIVE not in uploaded:
    raise FileNotFoundError(
        f"Expected {EXPECTED_ARCHIVE!r}; uploaded: {sorted(uploaded)}"
    )

archive_path = Path("/content") / EXPECTED_ARCHIVE
archive_path.write_bytes(uploaded[EXPECTED_ARCHIVE])
project_dir = Path("/content/rocmpilot")
if project_dir.exists():
    shutil.rmtree(project_dir)
project_dir.mkdir(parents=True)

with zipfile.ZipFile(archive_path) as archive:
    root = project_dir.resolve()
    for member in archive.infolist():
        destination = (project_dir / member.filename).resolve()
        if destination != root and root not in destination.parents:
            raise RuntimeError(f"Unsafe archive member: {member.filename}")
    archive.extractall(project_dir)

required_paths = [
    project_dir / "training/evaluate_benchmark.py",
    project_dir / "data/challenge_eval.jsonl",
    project_dir / "requirements-training.txt",
]
missing = [str(path) for path in required_paths if not path.is_file()]
if missing:
    raise FileNotFoundError(f"Archive is missing required project files: {missing}")

os.chdir(project_dir)
print("Project directory:", Path.cwd())


In [ ]:
# Cell 3 - Dependencies (preserve Colab's CUDA-enabled PyTorch)
import importlib
import re
import subprocess
import sys
from pathlib import Path

import torch

torch_version_before = torch.__version__
if not torch.cuda.is_available():
    raise RuntimeError("CUDA PyTorch disappeared; re-enable a Colab GPU runtime.")

source_requirements = Path("requirements-training.txt").read_text(encoding="utf-8").splitlines()
requirements_without_torch = [
    line for line in source_requirements
    if not re.match(r"^\s*torch(?:\s|[<=>~!\[]|$)", line, flags=re.IGNORECASE)
]
filtered_requirements = Path("/tmp/rocmpilot-requirements-without-torch.txt")
filtered_requirements.write_text("\n".join(requirements_without_torch) + "\n", encoding="utf-8")
experiment_1_pins = [
    "transformers==5.15.0",
    "accelerate==1.14.0",
    "peft==0.20.0",
    "datasets==4.0.0",
    "huggingface_hub==1.27.0",
    "sentencepiece==0.2.2",
    "protobuf==5.29.6",
    "safetensors==0.8.0",
    "torchao==0.18.0",
]
print("Installing non-Torch requirements with the versions recorded for Experiment 1:")
print(filtered_requirements.read_text(encoding="utf-8"))
print("Compatibility pins:", experiment_1_pins)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--upgrade-strategy", "only-if-needed", "-r", str(filtered_requirements), *experiment_1_pins],
    check=True,
)
importlib.invalidate_caches()

import accelerate
import peft
import transformers

if not torch.cuda.is_available():
    raise RuntimeError("Dependency installation left PyTorch without CUDA support. Restart with a GPU runtime.")
print("torch:", torch.__version__, "(before install:", torch_version_before + ")")
print("transformers:", transformers.__version__)
print("peft:", peft.__version__)
print("accelerate:", accelerate.__version__)


In [ ]:
# Cell 4 - Experiment metadata
import platform
import subprocess
import sys
from pathlib import Path

import accelerate
import peft
import torch
import transformers

reports_dir = Path("reports")
reports_dir.mkdir(exist_ok=True)
nvidia_smi_text = subprocess.run(
    ["nvidia-smi"], capture_output=True, text=True, check=True
).stdout
gpu_name = torch.cuda.get_device_name(0)
vram_gib = torch.cuda.get_device_properties(0).total_memory / 1024**3

revision_lines = []
try:
    from huggingface_hub import model_info
    for label, repo_id in [
        ("Base model", "Qwen/Qwen2.5-Coder-1.5B-Instruct"),
        ("LoRA adapter", "MrazzKa/rocmpilot-qwen25-coder-lora"),
    ]:
        revision_lines.append(f"{label} resolved main revision: {model_info(repo_id).sha}")
except Exception as exc:
    revision_lines.append(f"Hugging Face revision lookup unavailable (non-fatal): {type(exc).__name__}: {exc}")

hardware_text = (
    f"GPU model: {gpu_name}\n"
    f"Total VRAM (GiB): {vram_gib:.2f}\n"
    f"PyTorch CUDA runtime: {torch.version.cuda}\n\n"
    f"nvidia-smi\n{'=' * 80}\n{nvidia_smi_text}"
)
pip_freeze = subprocess.run(
    [sys.executable, "-m", "pip", "freeze"], capture_output=True, text=True, check=True
).stdout
environment_text = "\n".join([
    f"Platform: {platform.platform()}",
    f"Python: {sys.version.replace(chr(10), ' ')}",
    f"PyTorch: {torch.__version__}",
    f"CUDA available: {torch.cuda.is_available()}",
    f"PyTorch CUDA runtime: {torch.version.cuda}",
    f"Transformers: {transformers.__version__}",
    f"PEFT: {peft.__version__}",
    f"Accelerate: {accelerate.__version__}",
    f"GPU model: {gpu_name}",
    f"Total VRAM (GiB): {vram_gib:.2f}",
    *revision_lines,
    "",
    "pip freeze",
    "=" * 80,
    pip_freeze,
])
(reports_dir / "evaluation_hardware.txt").write_text(hardware_text, encoding="utf-8")
(reports_dir / "evaluation_environment.txt").write_text(environment_text, encoding="utf-8")
print(hardware_text)
print(environment_text.split("pip freeze", 1)[0])
print("Saved experiment metadata under reports/.")


In [ ]:
# Cell 5 - Smoke test: exactly one challenge example
import hashlib
import json
import subprocess
import sys
from pathlib import Path

from IPython.display import Markdown, display

challenge_path = Path("data/challenge_eval.jsonl")
CHALLENGE_SHA256_BEFORE_INFERENCE = hashlib.sha256(challenge_path.read_bytes()).hexdigest()
challenge_count = sum(1 for line in challenge_path.read_text(encoding="utf-8").splitlines() if line.strip())
if challenge_count != 8:
    raise RuntimeError(f"Expected 8 challenge examples, found {challenge_count}.")

smoke_command = [
    sys.executable, "training/evaluate_benchmark.py",
    "--base-model", "Qwen/Qwen2.5-Coder-1.5B-Instruct",
    "--adapter", "MrazzKa/rocmpilot-qwen25-coder-lora",
    "--dataset", "data/challenge_eval.jsonl",
    "--output-dir", "reports/smoke",
    "--device", "cuda",
    "--max-new-tokens", "512",
    "--limit", "1",
    "--seed", "42",
]
print("Running:", " ".join(smoke_command))
subprocess.run(smoke_command, check=True)

smoke_predictions = [
    json.loads(line)
    for line in Path("reports/smoke/benchmark_predictions.jsonl").read_text(encoding="utf-8").splitlines()
    if line.strip()
]
if len(smoke_predictions) != 1:
    raise RuntimeError(f"Smoke test should produce one prediction, got {len(smoke_predictions)}.")
smoke_results = json.loads(Path("reports/smoke/benchmark_results.json").read_text(encoding="utf-8"))
if smoke_results.get("experiment", {}).get("adapter") != "MrazzKa/rocmpilot-qwen25-coder-lora":
    raise RuntimeError("Smoke result does not identify the requested LoRA adapter.")

row = smoke_predictions[0]
display(Markdown("### Prompt"))
print(row["prompt"])
display(Markdown("### Base generation"))
print(row["base_generation"])
display(Markdown("### LoRA generation"))
print(row["adapter_generation"])
display(Markdown("### Metrics"))
print(json.dumps(row["metrics"], ensure_ascii=False, indent=2))
print("\nAdapter load evidence: the PEFT-backed adapter pass completed and was written to the result.")
print("Base and LoRA outputs are identical:", row["base_generation"].strip() == row["adapter_generation"].strip())
print("Dataset SHA-256 locked for the full run:", CHALLENGE_SHA256_BEFORE_INFERENCE)


## Cell 6 - Explicit confirmation point

> Inspect the smoke-test outputs above. Only continue if both base and LoRA generations are valid and the adapter loaded successfully.

Do not run the next cell if loading failed, either generation is empty or malformed, or the result does not clearly represent the requested base model and LoRA adapter. Identical text is not automatically an error, but it requires careful inspection before continuing.

In [ ]:
# Cell 7 - Full benchmark: all 8 unchanged challenge examples
import hashlib
import json
import subprocess
import sys
from pathlib import Path

challenge_path = Path("data/challenge_eval.jsonl")
current_hash = hashlib.sha256(challenge_path.read_bytes()).hexdigest()
if current_hash != CHALLENGE_SHA256_BEFORE_INFERENCE:
    raise RuntimeError("data/challenge_eval.jsonl changed after inference began; aborting.")

full_command = [
    sys.executable, "training/evaluate_benchmark.py",
    "--base-model", "Qwen/Qwen2.5-Coder-1.5B-Instruct",
    "--adapter", "MrazzKa/rocmpilot-qwen25-coder-lora",
    "--dataset", "data/challenge_eval.jsonl",
    "--output-dir", "reports",
    "--device", "cuda",
    "--max-new-tokens", "512",
    "--seed", "42",
]
print("Running:", " ".join(full_command))
subprocess.run(full_command, check=True)

if hashlib.sha256(challenge_path.read_bytes()).hexdigest() != CHALLENGE_SHA256_BEFORE_INFERENCE:
    raise RuntimeError("Challenge dataset changed during the full benchmark.")
full_results = json.loads(Path("reports/benchmark_results.json").read_text(encoding="utf-8"))
if full_results.get("status") != "completed" or full_results.get("number_of_examples") != 8:
    raise RuntimeError(f"Expected 8 completed examples, got: {full_results.get('number_of_examples')}")
print("Full benchmark completed for 8 examples; challenge SHA-256 remained unchanged.")


In [ ]:
# Cell 8 - Results summary (descriptive metrics, no automatic winner)
import json
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

results = json.loads(Path("reports/benchmark_results.json").read_text(encoding="utf-8"))
predictions = [
    json.loads(line)
    for line in Path("reports/benchmark_predictions.jsonl").read_text(encoding="utf-8").splitlines()
    if line.strip()
]
if len(predictions) != 8:
    raise RuntimeError(f"Expected 8 prediction rows, found {len(predictions)}.")

overall = results["overall"]
metric_names = list(overall["base"].keys())
aggregate_table = pd.DataFrame([
    {
        "Metric": metric,
        "Base score": overall["base"][metric],
        "LoRA score": overall["adapter"][metric],
        "Difference (LoRA - Base)": overall["difference_adapter_minus_base"][metric],
    }
    for metric in metric_names
])
display(Markdown("## Aggregate metrics"))
display(aggregate_table.style.format({"Base score": "{:.4f}", "LoRA score": "{:.4f}", "Difference (LoRA - Base)": "{:+.4f}"}))
display(Markdown("Lexical metrics are descriptive and do not determine a winner or prove technical correctness."))

for index, row in enumerate(predictions, 1):
    display(Markdown(f"## {index}. `{row['id']}` — {row['category']}"))
    display(Markdown("### Base generation"))
    print(row["base_generation"])
    display(Markdown("### LoRA generation"))
    print(row["adapter_generation"])
    display(Markdown("### Metric breakdown"))
    print(json.dumps(row["metrics"], ensure_ascii=False, indent=2))


In [ ]:
# Cell 9 - Manual-review helper; fill Human verdict yourself
import pandas as pd
from IPython.display import Markdown, display

manual_review_rows = []
for row in predictions:
    base = row["metrics"]["base"]
    adapter = row["metrics"]["adapter"]
    manual_review_rows.append({
        "Example ID": row["id"],
        "Category": row["category"],
        "Required concepts — Base": base["required_concept_coverage"],
        "Required concepts — LoRA": adapter["required_concept_coverage"],
        "Structural — Base": base["structural_compliance"],
        "Structural — LoRA": adapter["structural_compliance"],
        "Forbidden avoidance — Base": base["forbidden_concept_avoidance"],
        "Forbidden avoidance — LoRA": adapter["forbidden_concept_avoidance"],
        "Human verdict": "",
    })
manual_review = pd.DataFrame(manual_review_rows)
display(Markdown("Allowed manual values: **Base**, **LoRA**, **Tie**, or **Unclear**. No verdict is assigned automatically."))
display(manual_review)


In [ ]:
# Cell 10 - Package and download the actual benchmark results
import zipfile
from pathlib import Path

from google.colab import files

result_files = [
    Path("reports/benchmark_predictions.jsonl"),
    Path("reports/benchmark_results.json"),
    Path("reports/benchmark_results.csv"),
    Path("reports/eval_results.md"),
    Path("reports/evaluation_hardware.txt"),
    Path("reports/evaluation_environment.txt"),
]
missing = [str(path) for path in result_files if not path.is_file()]
if missing:
    raise FileNotFoundError(f"Cannot package results; missing files: {missing}")

results_archive = Path("/content/rocmpilot-benchmark-results.zip")
with zipfile.ZipFile(results_archive, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in result_files:
        archive.write(path, arcname=path.name)
print("Created:", results_archive)
print("Contents:", [path.name for path in result_files])
files.download(str(results_archive))
